In [1]:
%load_ext autoreload
%autoreload 2
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [2]:
import mediapipe as mp
import cv2
import numpy as np

In [4]:
from mediapipe.framework.formats import landmark_pb2
def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  # Loop through the detected poses to visualize.
  for idx in range(len(pose_landmarks_list)):
    pose_landmarks = pose_landmarks_list[idx]

    # Draw the pose landmarks.
    pose_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
    pose_landmarks_proto.landmark.extend([
      landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in pose_landmarks
    ])
    mp.solutions.drawing_utils.draw_landmarks(
      annotated_image,
      pose_landmarks_proto,
      mp.solutions.pose.POSE_CONNECTIONS,
      mp.solutions.drawing_styles.get_default_pose_landmarks_style())
  return annotated_image

In [5]:
%reload_ext autoreload
from src.shared.utils.video import frame_reader

# Now do multiple pose inference using mediapipe's new API
pose_options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path="../../models/mediapipe/pose_landmarker_heavy.task"
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_poses=4
)

with vision.PoseLandmarker.create_from_options(pose_options) as pose:
    for frame, timestamp in frame_reader(cap, fps=6, return_timestamp=True):
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        pose_landmarks = pose.detect_for_video(mp_image, timestamp)

        img = draw_landmarks_on_image(frame, pose_landmarks)
        cv2.imshow("image", img)
        cv2.waitKey(1)

I0000 00:00:1765293891.169224  335163 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765293891.172959  335395 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.3-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.48-1-MANJARO)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1765293891.210440  335401 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765293891.267959  335412 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/pipev/Documentos/Proyectos/SignTranslator/LSA-X/.venv/lib/python3.12/site-packages/cv2/qt/plugins"
W0000 00:00:1765293891.847668  335419 landmark_projection_calculator.cc:186] Using NORM_RECT witho

KeyboardInterrupt: 

In [6]:
# El multi pose estimation de mediapipe es un asco, voy a probar con YoloV11
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

True

In [7]:
from ultralytics import NAS
model = NAS("../../models/yolo/yolo_nas_m.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/pipev/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ModuleNotFoundError: No module named 'super_gradients'